In [ ]:
# ============================================================
# CELL 0 — Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 1 — List Sensor Files
# ============================================================

import os
SENSOR_DIR = "/content/drive/MyDrive/data_collection/sensor_data"
files = sorted(os.listdir(SENSOR_DIR))
print(f"Total files: {len(files)}")
for f in files:
    print(f"  {f}")

In [ ]:
# ============================================================
# CELL 2 — AVI → MP4 Compression
# ============================================================

import glob
import os

input_folder = "/content/drive/MyDrive/data_collection/camera/*.avi"

for file in glob.glob(input_folder):
    output = file.replace(".avi", "_compressed.mp4")
    command = f'ffmpeg -i "{file}" -vcodec libx264 -crf 23 "{output}"'
    print("Converting:", os.path.basename(file))
    os.system(command)

In [ ]:
# ============================================================
# CELL 3 — normalize_columns() — old D1–D9 ↔ new AX/AY/AZ/ROLL/PITCH/YAW/LAT/LON
# ============================================================

import pandas as pd
import numpy as np

def normalize_columns(df):
    """
    Detect old format (D1-D6) vs new format (AX/AY/AZ/ROLL/PITCH/YAW)
    and remap to a unified internal schema.

    Old fusion format columns:
        TYPE, TIME, D1, D2, D3, D4, D5, D6, D7, D8, D9
        IMU rows: D1=AX, D2=AY, D3=AZ, D4=ROLL, D5=PITCH, D6=YAW
        GPS rows: D1=LAT, D2=LON, D3=ALT, D4=SPEED_KMH, D5=COURSE

    New fusion format columns:
        TYPE, TIME, AX, AY, AZ, ROLL, PITCH, YAW, LAT, LON, SPEED_KMH, FRAME_ID
    """
    df.columns = [c.upper() for c in df.columns]

    # Detect format by checking if D1 exists
    if "D1" in df.columns:
        # ── OLD FORMAT ────────────────────────────────────────────────
        # IMU rows: remap D1-D6 to named columns
        imu_mask = df["TYPE"] == "IMU"
        gps_mask = df["TYPE"] == "GPS"

        # Initialize new columns with NaN
        for col in ["AX", "AY", "AZ", "ROLL", "PITCH", "YAW",
                    "LAT", "LON", "SPEED_KMH", "FRAME_ID"]:
            if col not in df.columns:
                df[col] = np.nan

        # IMU rows: D1=AX, D2=AY, D3=AZ, D4=ROLL, D5=PITCH, D6=YAW
        df.loc[imu_mask, "AX"]    = pd.to_numeric(df.loc[imu_mask, "D1"], errors="coerce")
        df.loc[imu_mask, "AY"]    = pd.to_numeric(df.loc[imu_mask, "D2"], errors="coerce")
        df.loc[imu_mask, "AZ"]    = pd.to_numeric(df.loc[imu_mask, "D3"], errors="coerce")
        df.loc[imu_mask, "ROLL"]  = pd.to_numeric(df.loc[imu_mask, "D4"], errors="coerce")
        df.loc[imu_mask, "PITCH"] = pd.to_numeric(df.loc[imu_mask, "D5"], errors="coerce")
        df.loc[imu_mask, "YAW"]   = pd.to_numeric(df.loc[imu_mask, "D6"], errors="coerce")

        # GPS rows: D1=LAT, D2=LON, D4=SPEED (knots in old format → convert)
        df.loc[gps_mask, "LAT"] = pd.to_numeric(df.loc[gps_mask, "D1"], errors="coerce")
        df.loc[gps_mask, "LON"] = pd.to_numeric(df.loc[gps_mask, "D2"], errors="coerce")

        # Old format stored speed in knots — convert to km/h
        if "D4" in df.columns:
            raw_speed = pd.to_numeric(df.loc[gps_mask, "D4"], errors="coerce")
            df.loc[gps_mask, "SPEED_KMH"] = raw_speed * 1.852

        # CAM rows: D1 or D9 = FRAME_ID depending on old script version
        cam_mask = df["TYPE"] == "CAM"
        if "D9" in df.columns:
            df.loc[cam_mask, "FRAME_ID"] = pd.to_numeric(df.loc[cam_mask, "D9"], errors="coerce")

        print("  [format] Old D1-D6 detected → remapped to named columns")

    else:
        # ── NEW FORMAT ────────────────────────────────────────────────
        # Columns already named correctly, just ensure numeric types
        for col in ["AX", "AY", "AZ", "ROLL", "PITCH", "YAW",
                    "LAT", "LON", "SPEED_KMH"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        print("  [format] New named-column format detected ✓")

    return df

In [ ]:
# ============================================================
# CELL 4 — GPS Frame Extractor — handles separate / sensor / fusion CSV formats
# ============================================================

import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

DRIVE_BASE  = "/content/drive/MyDrive/data_collection"
VIDEO_DIR   = os.path.join(DRIVE_BASE, "camera")
SENSOR_DIR  = os.path.join(DRIVE_BASE, "sensor_data")
FRAMES_DIR  = os.path.join(DRIVE_BASE, "frames")
os.makedirs(FRAMES_DIR, exist_ok=True)

SKIP_SESSIONS = {"20260402_164557"}

# ─────────────────────────────────────────
# FIND VIDEO
# ─────────────────────────────────────────
def find_video(session_id):
    for name in [
        f"video_{session_id}_compressed.mp4",
        f"video_{session_id}.mp4",
        f"video_{session_id}.avi",
    ]:
        p = os.path.join(VIDEO_DIR, name)
        if os.path.exists(p):
            return p
    return None

# ─────────────────────────────────────────
# FIND AND LOAD CSV — handles all 3 formats
# Returns: (gps_df, cam_df, format_name)
# ─────────────────────────────────────────
def load_sensor_data(session_id):

    # ── Format 3: fusion_*.csv ───────────────────────────────
    fusion_path = os.path.join(SENSOR_DIR, f"fusion_{session_id}.csv")
    if os.path.exists(fusion_path):
        df = pd.read_csv(fusion_path)
        df.columns = [c.strip().upper() for c in df.columns]
        df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")

        # Handle old D1/D2 column names if present
        if "D1" in df.columns and "LAT" not in df.columns:
            gps_mask = df["TYPE"] == "GPS"
            df.loc[gps_mask, "LAT"] = pd.to_numeric(df.loc[gps_mask, "D1"], errors="coerce")
            df.loc[gps_mask, "LON"] = pd.to_numeric(df.loc[gps_mask, "D2"], errors="coerce")

        if "FRAME_ID" not in df.columns:
            df["FRAME_ID"] = None
            cam_mask = df["TYPE"] == "CAM"
            df.loc[cam_mask, "FRAME_ID"] = list(range(cam_mask.sum()))

        gps_df = df[df["TYPE"] == "GPS"].copy().reset_index(drop=True)
        cam_df = df[df["TYPE"] == "CAM"].copy().reset_index(drop=True)
        return gps_df, cam_df, "fusion"

    # ── Format 2: sensor_*.csv ───────────────────────────────
    sensor_path = os.path.join(SENSOR_DIR, f"sensor_{session_id}.csv")
    if os.path.exists(sensor_path):
        df = pd.read_csv(sensor_path)
        df.columns = [c.strip().upper() for c in df.columns]
        df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")

        # sensor CSV uses D1-D6, TYPE column same
        if "D1" in df.columns and "LAT" not in df.columns:
            gps_mask = df["TYPE"] == "GPS"
            df.loc[gps_mask, "LAT"] = pd.to_numeric(df.loc[gps_mask, "D1"], errors="coerce")
            df.loc[gps_mask, "LON"] = pd.to_numeric(df.loc[gps_mask, "D2"], errors="coerce")

        if "FRAME_ID" not in df.columns:
            df["FRAME_ID"] = None
            cam_mask = df["TYPE"] == "CAM"
            df.loc[cam_mask, "FRAME_ID"] = list(range(cam_mask.sum()))

        gps_df = df[df["TYPE"] == "GPS"].copy().reset_index(drop=True)
        cam_df = df[df["TYPE"] == "CAM"].copy().reset_index(drop=True)
        return gps_df, cam_df, "sensor"

    # ── Format 1: separate gps_*.csv + imu_*.csv ────────────
    gps_path = os.path.join(SENSOR_DIR, f"gps_{session_id}.csv")
    if os.path.exists(gps_path):
        gps_raw = pd.read_csv(gps_path)
        gps_raw.columns = [c.strip().upper() for c in gps_raw.columns]
        gps_raw["TIME"] = pd.to_datetime(gps_raw["TIME"], errors="coerce")

        # Old GPS CSV: columns are TIME, LAT, LON, ALT, SPEED etc
        # Normalize to LAT/LON if named differently
        for old, new in [("LATITUDE","LAT"),("LONGITUDE","LON")]:
            if old in gps_raw.columns and new not in gps_raw.columns:
                gps_raw.rename(columns={old: new}, inplace=True)

        gps_raw["TYPE"]     = "GPS"
        gps_raw["FRAME_ID"] = None

        # No CAM rows in old format — build synthetic ones from GPS timestamps
        # Each GPS fix maps to nearest video frame by time offset from session start
        gps_raw = gps_raw.dropna(subset=["TIME"]).sort_values("TIME").reset_index(drop=True)

        if not gps_raw.empty:
            t0 = gps_raw["TIME"].iloc[0]
            gps_raw["TIME_OFFSET"] = (gps_raw["TIME"] - t0).dt.total_seconds()

        # Build synthetic CAM df — one row per second of video
        # Will be matched by time offset
        cam_df = gps_raw[["TIME"]].copy()
        cam_df["TYPE"]     = "CAM"
        cam_df["FRAME_ID"] = None  # will be computed from time offset

        return gps_raw, cam_df, "separate"

    return None, None, None

# ─────────────────────────────────────────
# EXTRACT FRAMES FOR ONE SESSION
# ─────────────────────────────────────────
def extract_frames_for_session(session_id):
    out_dir = os.path.join(FRAMES_DIR, session_id)
    os.makedirs(out_dir, exist_ok=True)

    # ── CHECKPOINT CHECK ─────────────────────────────────────
    # If frames already exist for this session, skip it
    existing_frames = [f for f in os.listdir(out_dir) if f.endswith(".jpg")]
    if len(existing_frames) > 0:
        print(f"  [SKIP] {session_id} — already extracted ({len(existing_frames)} frames)")
        return len(existing_frames)

    video_path = find_video(session_id)
    if video_path is None:
        print(f"  [SKIP] {session_id} — no video found")
        return 0

    gps_df, cam_df, fmt = load_sensor_data(session_id)
    if gps_df is None:
        print(f"  [SKIP] {session_id} — no sensor data found")
        return 0
    if gps_df.empty:
        print(f"  [SKIP] {session_id} — no GPS rows")
        return 0

    print(f"  [{fmt.upper()}] {session_id} — {len(gps_df)} GPS fixes")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  [SKIP] {session_id} — could not open video")
        return 0

    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps           = cap.get(cv2.CAP_PROP_FPS) or 30.0
    saved         = 0
    session_start = gps_df["TIME"].dropna().min()

    for _, gps_row in tqdm(gps_df.iterrows(),
                           total=len(gps_df),
                           desc=f"  {session_id}",
                           leave=False):

        gps_time = gps_row["TIME"]
        if pd.isnull(gps_time):
            continue

        lat = float(gps_row.get("LAT", 0) or 0)
        lon = float(gps_row.get("LON", 0) or 0)
        if lat == 0.0 and lon == 0.0:
            continue

        frame_id = None

        if fmt in ("fusion", "sensor"):
            try:
                gps_frame_val = gps_row.get("FRAME_ID", None)
                if pd.notna(gps_frame_val) and str(gps_frame_val).strip() not in ("", "None", "nan"):
                    frame_id = int(float(gps_frame_val))
            except (ValueError, TypeError):
                frame_id = None

            if frame_id is None and not cam_df.empty:
                time_diffs  = (cam_df["TIME"] - gps_time).abs()
                nearest_idx = time_diffs.idxmin()
                diff_sec    = time_diffs[nearest_idx].total_seconds()
                if diff_sec <= 1.0:
                    try:
                        frame_id = int(float(cam_df.loc[nearest_idx, "FRAME_ID"]))
                    except (ValueError, TypeError):
                        frame_id = None

        if fmt == "separate":
            if not pd.isnull(session_start):
                offset_sec = (gps_time - session_start).total_seconds()
                frame_id   = int(offset_sec * fps)

        if frame_id is None:
            continue

        frame_id = max(0, min(frame_id, total_frames - 1))

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            continue

        out_name = f"frame_{frame_id:06d}_lat{round(lat,7)}_lon{round(lon,7)}.jpg"
        cv2.imwrite(os.path.join(out_dir, out_name), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 85])
        saved += 1

    cap.release()
    return saved

# ─────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────
import re

sessions = sorted(set(
    re.sub(r"^video_", "", re.sub(r"(_compressed)?\.mp4$", "", f))
    for f in os.listdir(VIDEO_DIR)
    if f.endswith(".mp4")
) - SKIP_SESSIONS)

print(f"Found {len(sessions)} sessions\n")
total_saved  = 0
total_skipped = 0

for session_id in sessions:
    n = extract_frames_for_session(session_id)
    if n > 0:
        existing = [f for f in os.listdir(os.path.join(FRAMES_DIR, session_id))
                    if f.endswith(".jpg")]
        if len(existing) == n and n > 0:
            total_skipped += 1
        else:
            total_saved += n

print(f"\nDone.")
print(f"Output: {FRAMES_DIR}")

In [ ]:
# ============================================================
# CELL 5 — Heatmap Window Frame Extractor
# ============================================================

import os
import cv2
import pandas as pd
from tqdm import tqdm

DRIVE_BASE  = "/content/drive/MyDrive/data_collection"
FRAMES_DIR  = os.path.join(DRIVE_BASE, "frames", "heatmap_windows")
SENSOR_DIR  = os.path.join(DRIVE_BASE, "sensor_data")
VIDEO_DIR   = os.path.join(DRIVE_BASE, "camera")
OUT_DIR     = "/content/drive/MyDrive/data_collection/rskid_output"

os.makedirs(FRAMES_DIR, exist_ok=True)

# ─────────────────────────────────────────
# LOAD R_SKID DATASET
# ─────────────────────────────────────────
full_df = pd.read_csv(f"{OUT_DIR}/rskid_all_sessions.csv")
full_df.columns = [c.upper() for c in full_df.columns]
print(f"Loaded {len(full_df)} windows across {full_df['SESSION'].nunique()} sessions\n")

WINDOW_SEC = 2.0

def find_video(session_id):
    for name in [
        f"video_{session_id}_compressed.mp4",
        f"video_{session_id}.mp4",
    ]:
        p = os.path.join(VIDEO_DIR, name)
        if os.path.exists(p):
            return p
    return None

def find_fusion(session_id):
    for prefix in ["fusion", "sensor"]:
        p = os.path.join(SENSOR_DIR, f"{prefix}_{session_id}.csv")
        if os.path.exists(p):
            return p
    return None

# ─────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────
saved_total = 0
skipped     = 0

for session_ts, group in full_df.groupby("SESSION"):

    fusion_path = find_fusion(session_ts)
    video_path  = find_video(session_ts)

    if fusion_path is None or video_path is None:
        print(f"  [SKIP] {session_ts} — missing fusion or video")
        skipped += len(group)
        continue

    # ─────────────────────────────────────
    # CHECKPOINT CHECK
    # FIX: compare against existing frame count, not group size.
    # group size changes every time Cell 13 is re-run (window count
    # shifts due to motion gate / noise floor), but the frames
    # themselves never need re-extraction unless you delete the folder.
    # ─────────────────────────────────────
    out_dir  = os.path.join(FRAMES_DIR, session_ts)
    os.makedirs(out_dir, exist_ok=True)
    existing = [f for f in os.listdir(out_dir) if f.endswith(".jpg")]

    if len(existing) > 0:
        print(f"  [SKIP] {session_ts} — already done ({len(existing)} frames)")
        saved_total += len(existing)
        continue

    # Load fusion CSV
    fdf = pd.read_csv(fusion_path)
    fdf.columns = [c.strip().upper() for c in fdf.columns]
    fdf["TIME"] = pd.to_datetime(fdf["TIME"], errors="coerce")
    fdf = fdf.dropna(subset=["TIME"]).sort_values("TIME").reset_index(drop=True)

    # Handle old D1/D2 GPS columns
    if "D1" in fdf.columns and "LAT" not in fdf.columns:
        gps_mask = fdf["TYPE"] == "GPS"
        fdf.loc[gps_mask, "LAT"] = pd.to_numeric(fdf.loc[gps_mask, "D1"], errors="coerce")
        fdf.loc[gps_mask, "LON"] = pd.to_numeric(fdf.loc[gps_mask, "D2"], errors="coerce")

    # Get GPS and CAM rows
    gps_rows = fdf[fdf["TYPE"] == "GPS"].copy().reset_index(drop=True)
    cam_rows = fdf[fdf["TYPE"] == "CAM"].copy().reset_index(drop=True)

    # Sequential frame index — this is the frame number to seek in video
    cam_rows["FRAME_IDX"] = range(len(cam_rows))

    gps_rows = gps_rows[
        gps_rows["LAT"].notna() & gps_rows["LON"].notna() &
        (gps_rows["LAT"] != 0)  & (gps_rows["LON"] != 0)
    ]

    if gps_rows.empty or cam_rows.empty:
        print(f"  [SKIP] {session_ts} — no GPS or CAM rows")
        skipped += len(group)
        continue

    # Session start = first row timestamp
    t0 = fdf["TIME"].iloc[0]

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  [SKIP] {session_ts} — cannot open video")
        skipped += len(group)
        continue

    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved_session = 0

    for _, row in tqdm(group.iterrows(),
                       total=len(group),
                       desc=f"  {session_ts}",
                       leave=False):

        wid    = int(row["WINDOW_ID"])
        r_skid = row["R_SKID_FINAL"]
        risk   = row["RISK_CLASS"]

        t_start = t0 + pd.Timedelta(seconds=wid * WINDOW_SEC)
        t_end   = t_start + pd.Timedelta(seconds=WINDOW_SEC)
        t_mid   = t_start + pd.Timedelta(seconds=WINDOW_SEC / 2)

        # ── Get GPS for this window ───────────────────────────
        gps_win = gps_rows[
            (gps_rows["TIME"] >= t_start) &
            (gps_rows["TIME"] <  t_end)
        ]
        if gps_win.empty:
            skipped += 1
            continue

        lat = round(gps_win["LAT"].mean(), 7)
        lon = round(gps_win["LON"].mean(), 7)

        # ── Get nearest CAM row to window midpoint ────────────
        time_diffs  = (cam_rows["TIME"] - t_mid).abs()
        nearest_idx = time_diffs.idxmin()
        frame_id    = int(cam_rows.loc[nearest_idx, "FRAME_IDX"])
        frame_id    = max(0, min(frame_id, total_frames - 1))

        # ── Seek and extract ──────────────────────────────────
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            skipped += 1
            continue

        risk_short = {"Safe": "S", "Caution": "C", "High Risk": "HR"}.get(risk, "U")
        out_name   = (f"w{wid:04d}_"
                      f"r{r_skid:.3f}_"
                      f"{risk_short}_"
                      f"lat{lat}_lon{lon}.jpg")

        cv2.imwrite(os.path.join(out_dir, out_name), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 85])
        saved_session += 1

    cap.release()
    saved_total += saved_session
    print(f"  ✅ {session_ts} — {saved_session}/{len(group)} windows saved")

print(f"\n{'='*50}")
print(f"✅ Done. {saved_total} frames saved")
print(f"   Skipped: {skipped} windows (no GPS or no CAM match)")
print(f"   Output: {FRAMES_DIR}")
print(f"\nFilename format: w0042_r0.412_C_lat13.123_lon80.456.jpg")
print(f"                  └wid  └r_skid └S/C/HR  └GPS coords")

In [ ]:
# ============================================================
# CELL 6 — Fusion CSV Explorer + GPS Folium Map
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from pathlib import Path
from IPython.display import display

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
BASE_DIR   = "/content/drive/MyDrive/data_collection"
SENSOR_DIR = f"{BASE_DIR}/sensor_data"
FILES      = sorted(Path(SENSOR_DIR).glob("fusion_*.csv"))
print(f"Found {len(FILES)} fusion files")

def load_fusion(path):
    df = pd.read_csv(path)
    df = normalize_columns(df)
    df.columns = [c.upper() for c in df.columns]
    df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")
    df = df.dropna(subset=["TIME"]).sort_values("TIME")
    return df

def split(df):
    return {
        "IMU": df[df["TYPE"] == "IMU"].copy(),
        "GPS": df[df["TYPE"] == "GPS"].copy(),
        "CAM": df[df["TYPE"] == "CAM"].copy(),
    }

# ═════════════════════════════════════════
# PART A — FUSION CSV EXPLORER (summary + IMU/GPS/CAM plots)
# ═════════════════════════════════════════
def summarize(name, df, parts):
    print(f"\n{'='*50}")
    print(f" SUMMARY → {name}")
    print(f"{'='*50}")
    print(f"Total rows : {len(df)}")
    print(f"IMU rows   : {len(parts['IMU'])}")
    print(f"GPS rows   : {len(parts['GPS'])}")
    print(f"CAM rows   : {len(parts['CAM'])}")
    deltas = df["TIME"].diff().dropna()
    if not deltas.empty:
        print(f"Mean Δt    : {deltas.mean().total_seconds()*1000:.2f} ms")
        print(f"Max  Δt    : {deltas.max().total_seconds()*1000:.2f} ms")

def plot_imu(parts, name):
    imu = parts["IMU"]
    if imu.empty:
        print("No IMU data"); return
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    fig.suptitle(f"IMU → {name}")
    axes[0].plot(imu["TIME"], imu["AX"], label="Ax")
    axes[0].plot(imu["TIME"], imu["AY"], label="Ay")
    axes[0].plot(imu["TIME"], imu["AZ"], label="Az")
    axes[0].set_ylabel("m/s²"); axes[0].set_title("Linear Acceleration")
    axes[0].legend(); axes[0].grid()
    axes[1].plot(imu["TIME"], imu["ROLL"],  label="Roll")
    axes[1].plot(imu["TIME"], imu["PITCH"], label="Pitch")
    axes[1].plot(imu["TIME"], imu["YAW"],   label="Yaw")
    axes[1].set_ylabel("Degrees"); axes[1].set_title("Orientation")
    axes[1].legend(); axes[1].grid()
    plt.tight_layout(); plt.show()

def plot_gps_speed(parts, name):
    gps = parts["GPS"]
    if gps.empty:
        print("No GPS data"); return
    gps = gps.dropna(subset=["LAT", "LON"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"GPS → {name}")
    axes[0].plot(gps["LON"], gps["LAT"], marker="o", markersize=2)
    axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
    axes[0].set_title("GPS Trajectory"); axes[0].grid()
    if "SPEED_KMH" in gps.columns:
        speed = gps["SPEED_KMH"].replace(0, np.nan).dropna()
        if not speed.empty:
            axes[1].plot(gps.loc[speed.index, "TIME"], speed)
            axes[1].set_ylabel("Speed (km/h)")
            axes[1].set_title("GPS Speed over Time"); axes[1].grid()
        else:
            axes[1].set_title("No speed data yet (GPS fix needed)")
    else:
        axes[1].set_title("SPEED_KMH column not found")
    plt.tight_layout(); plt.show()

def plot_cam(parts, name):
    cam = parts["CAM"]
    if cam.empty:
        print("No CAM data"); return
    deltas = cam["TIME"].diff().dt.total_seconds().dropna()
    plt.figure(figsize=(10, 3))
    plt.hist(deltas, bins=40)
    plt.title(f"Camera Δt Distribution → {name}")
    plt.xlabel("Seconds"); plt.grid(); plt.show()

# ═════════════════════════════════════════
# PART B — GPS FOLIUM MAP (speed colour-coded)
# ═════════════════════════════════════════
def speed_color(speed_kmh):
    """Green = slow, Orange = moderate, Red = fast"""
    if speed_kmh < 15:
        return "green"
    elif speed_kmh < 35:
        return "orange"
    else:
        return "red"

def plot_gps_map(gps, name):
    gps = gps.dropna(subset=["LAT", "LON"])
    gps = gps[(gps["LAT"] != 0) & (gps["LON"] != 0)]
    if gps.empty:
        print(f"No valid GPS for {name}"); return None
    lat_c = gps["LAT"].mean(); lon_c = gps["LON"].mean()
    m = folium.Map(location=[lat_c, lon_c], zoom_start=17, tiles="OpenStreetMap")
    path = list(zip(gps["LAT"], gps["LON"]))
    folium.PolyLine(path, color="blue", weight=3, opacity=0.6).add_to(m)
    if "SPEED_KMH" in gps.columns:
        for _, row in gps.iloc[::5].iterrows():
            spd = row["SPEED_KMH"]
            if spd > 0:
                folium.CircleMarker(
                    location=[row["LAT"], row["LON"]],
                    radius=4, color=speed_color(spd),
                    fill=True, fill_opacity=0.8,
                    popup=f"{spd:.1f} km/h"
                ).add_to(m)
    folium.Marker(path[0],  popup="Start", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(path[-1], popup="End",   icon=folium.Icon(color="red")).add_to(m)
    return m

# ─────────────────────────────────────────
# MAIN LOOP — explore each fusion file, then render its GPS map
# ─────────────────────────────────────────
summary = {}
for path in FILES:
    df    = load_fusion(path)
    parts = split(df)
    summarize(path.name, df, parts)
    plot_imu(parts, path.name)
    plot_gps_speed(parts, path.name)
    plot_cam(parts, path.name)

    print(f"\n▶ GPS map: {path.name}  ({len(parts['GPS'])} GPS rows)")
    m = plot_gps_map(parts["GPS"], path.name)
    if m:
        display(m)

    summary[path.name] = {
        "total": len(df),
        "imu": len(parts["IMU"]),
        "gps": len(parts["GPS"]),
        "cam": len(parts["CAM"]),
    }

print("\n===== FINAL SUMMARY =====")
display(pd.DataFrame(summary).T)

In [ ]:
# ============================================================
# CELL 7 — pip install
# ============================================================

pip install opencv-python pandas numpy tqdm

In [ ]:
# ============================================================
# CELL 8 — Vision Feature Extraction — grey-world WB, mud_score s<220 (explicit column)
# ============================================================

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from pathlib import Path

# -----------------------------
# CONFIG
# -----------------------------
VIDEO_DIR = "/content/drive/MyDrive/data_collection/camera"
OUT_DIR = "/content/drive/MyDrive/data_collection/video_features"
CHECKPOINT_DIR = "/content/drive/MyDrive/data_collection/.ipynb_checkpoints"
FPS = 15
CHECKPOINT_INTERVAL = 500

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

VIDEOS = sorted(Path(VIDEO_DIR).glob("*.mp4"))
print(f"🎥 Found {len(VIDEOS)} videos")

# ─────────────────────────────────────────
# PER-SESSION ROI CONFIG
# ─────────────────────────────────────────
ROI_FULL = [
    "20250729163940", "20250729164433", "20251120111431",
    "20251205160559", "20251206154342", "20251207170709",
    "20260103130421", "20260103160251", "20260103160852",
    "20260103161424", "20260103162257", "20260408_180041",
    "20260408_180357", "20260408_182004", "20260408_182524",
    "20260420_133223", "20260420_133423", "20260420_134058",
    "20260420_134702", "20260420_135629"
]

ROI_80 = [
    "20250729154716", "20250730110618", "20250730110859",
    "20251206162234", "20251207152125", "20251207170351"
]

SKIP_SESSIONS   = ["20260402_164557"]
STATIC_SESSIONS = ["20260408_185057"]

def get_roi_for_session(session_ts):
    if session_ts in ROI_FULL:
        return 0.0, 1.0
    elif session_ts in ROI_80:
        return 0.2, 1.0
    else:
        return 0.4, 1.0

def get_road_roi(frame, session_ts):
    h, w = frame.shape[:2]
    top_ratio, bot_ratio = get_roi_for_session(session_ts)
    top = int(h * top_ratio)
    bot = int(h * bot_ratio)
    return frame[top:bot, :]

# ─────────────────────────────────────────
# EXPECTED COLUMNS
# ─────────────────────────────────────────
EXPECTED_COLUMNS = [
    'frame_id', 'time_sec',
    'flow_mag_mean', 'flow_mag_std', 'flow_mag_max',
    'edge_density', 'edge_magnitude_mean', 'edge_magnitude_std',
    'hue_mean', 'hue_std', 'saturation_mean', 'saturation_std',
    'value_mean', 'value_std',
    'specular_ratio', 'wetness_ratio', 'brightness_uniformity',
    'texture_roughness', 'texture_std',
    'lightness_mean', 'color_variance',
    'brightness_variance',
    'mud_score',
]

# -----------------------------
# FIX 1 — WHITE BALANCE
# Removes ambient color cast (orange sunset, tungsten light, overcast grey)
# so HSV thresholds see true surface color, not lighting color
# -----------------------------
def grey_world_wb(frame_bgr):
    """
    Grey-world white balance correction.
    Assumes the average color of the scene should be neutral grey.
    Scales each BGR channel so all three means become equal.
    This neutralizes orange evening light, yellow indoor lights, etc.
    """
    result = frame_bgr.copy().astype(np.float32)
    mean_b = np.mean(result[:, :, 0])
    mean_g = np.mean(result[:, :, 1])
    mean_r = np.mean(result[:, :, 2])
    mean_grey = (mean_b + mean_g + mean_r) / 3.0
    # Scale each channel so its mean = mean_grey
    result[:, :, 0] = np.clip(result[:, :, 0] * (mean_grey / (mean_b + 1e-6)), 0, 255)
    result[:, :, 1] = np.clip(result[:, :, 1] * (mean_grey / (mean_g + 1e-6)), 0, 255)
    result[:, :, 2] = np.clip(result[:, :, 2] * (mean_grey / (mean_r + 1e-6)), 0, 255)
    return result.astype(np.uint8)

# -----------------------------
# FEATURE EXTRACTION FUNCTIONS
# -----------------------------
def compute_optical_flow(prev_gray, curr_gray):
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return {
        'flow_mag_mean': np.mean(mag),
        'flow_mag_std':  np.std(mag),
        'flow_mag_max':  np.max(mag)
    }

def compute_edge_features(gray):
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_magnitude = np.sqrt(sobelx**2 + sobely**2)
    return {
        'edge_density':          edge_density,
        'edge_magnitude_mean':   np.mean(edge_magnitude),
        'edge_magnitude_std':    np.std(edge_magnitude)
    }

def compute_hsv_features(frame_roi_wb):
    # NOTE: receives white-balanced frame
    hsv = cv2.cvtColor(frame_roi_wb, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    return {
        'hue_mean':        np.mean(h),
        'hue_std':         np.std(h),
        'saturation_mean': np.mean(s),
        'saturation_std':  np.std(s),
        'value_mean':      np.mean(v),
        'value_std':       np.std(v)
    }

def compute_wetness_index(frame_roi_wb, gray_roi):
    """
    FIX 1: Receives white-balanced frame — color cast removed before HSV.
    FIX 2: mud threshold s<220 (was s<150 — missed real mud sat 150-200).
    FIX 3: mud_score saved as explicit column (was not being saved before).
    """
    hsv = cv2.cvtColor(frame_roi_wb, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Wet TAR — grey reflective asphalt
    wet_mask_tar   = (s < 50) & (v > 120)

    # Wet MUD — earthy hue, medium-high saturation, medium brightness
    # FIX: s<220 instead of old s<150
    wet_mask_mud   = (h > 10) & (h < 40) & (s > 40) & (s < 220) & (v > 80) & (v < 210)

    # Standing water — very low saturation, very high brightness
    wet_mask_water = (s < 30) & (v > 150)

    wetness_tar   = np.sum(wet_mask_tar)   / wet_mask_tar.size
    wetness_mud   = np.sum(wet_mask_mud)   / wet_mask_mud.size
    wetness_water = np.sum(wet_mask_water) / wet_mask_water.size
    wetness_ratio = max(wetness_tar, wetness_mud, wetness_water)

    bright_threshold      = np.percentile(gray_roi, 95)
    specular_ratio        = np.sum(gray_roi > bright_threshold) / gray_roi.size
    brightness_uniformity = 1.0 / (np.std(gray_roi) + 1)
    brightness_variance   = float(np.var(gray_roi))

    # FIX: mud_score explicitly returned as its own key
    mud_score = float(np.mean(wet_mask_mud.astype(np.float32)))

    return {
        'specular_ratio':         specular_ratio,
        'wetness_ratio':          wetness_ratio,
        'brightness_uniformity':  brightness_uniformity,
        'brightness_variance':    brightness_variance,
        'mud_score':              mud_score,   # ← was missing before
    }

def compute_texture_features(gray_roi):
    kernel_size = 15
    mean_filter = cv2.blur(gray_roi, (kernel_size, kernel_size))
    variance    = (gray_roi.astype(float) - mean_filter) ** 2
    local_std   = np.sqrt(cv2.blur(variance, (kernel_size, kernel_size)))
    return {
        'texture_roughness': np.mean(local_std),
        'texture_std':       np.std(local_std)
    }

def compute_color_features(frame_roi_wb):
    # NOTE: receives white-balanced frame
    lab = cv2.cvtColor(frame_roi_wb, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    return {
        'lightness_mean': np.mean(l),
        'color_variance': np.var(a) + np.var(b)
    }

# -----------------------------
# CHECKPOINT MANAGEMENT
# -----------------------------
def save_checkpoint(checkpoint_path, all_features):
    if all_features:
        pd.DataFrame(all_features).to_csv(checkpoint_path, index=False)

def load_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path):
        try:
            df = pd.read_csv(checkpoint_path)
            if set(EXPECTED_COLUMNS).issubset(set(df.columns)):
                return df
            else:
                print(f"   ⚠️  Checkpoint missing columns, starting fresh")
                return None
        except:
            return None
    return None

# -----------------------------
# PROCESS ONE VIDEO
# -----------------------------
def process_video(video_path):
    video_path  = str(video_path)
    video_name  = os.path.basename(video_path)
    base_name   = video_name.replace("_compressed", "").replace(".mp4", "")
    session_ts  = base_name.replace("video_", "")
    out_csv     = os.path.join(OUT_DIR, f"{base_name}_features.csv")
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{base_name}_checkpoint.csv")

    print(f"\n{'='*60}")
    print(f"▶ Processing: {video_name}")
    print(f"{'='*60}")

    if session_ts in SKIP_SESSIONS:
        print(f"⏭️  Skipping — marked as test/unusable")
        return

    top_ratio, _ = get_roi_for_session(session_ts)
    roi_label = "100%" if top_ratio == 0.0 else ("80%" if top_ratio == 0.2 else "60%")
    print(f"   ROI    : lower {roi_label} of frame")
    print(f"   WB     : grey-world white balance ON")  # FIX indicator

    if session_ts in STATIC_SESSIONS:
        print(f"   ⚠️  Static session — optical flow will be near zero")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Cannot open video")
        return

    fps_actual   = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration     = total_frames / fps_actual

    print(f"   FPS    : {fps_actual:.1f}")
    print(f"   Frames : {total_frames}")
    print(f"   Duration: {duration:.1f}s ({duration/60:.1f} min)")

    if os.path.exists(out_csv):
        existing_df     = pd.read_csv(out_csv)
        has_all_columns = set(EXPECTED_COLUMNS).issubset(set(existing_df.columns))
        has_all_frames  = len(existing_df) >= total_frames * 0.90
        if has_all_columns and has_all_frames:
            print(f"✅ Already processed ({len(existing_df)}/{total_frames} frames) — skipping")
            cap.release()
            return
        else:
            reason = "missing columns" if not has_all_columns else f"incomplete ({len(existing_df)}/{total_frames} frames)"
            print(f"⚠️  Re-extracting — {reason}")

    checkpoint_df = load_checkpoint(checkpoint_path)
    if checkpoint_df is not None and len(checkpoint_df) > 0:
        start_frame  = len(checkpoint_df)
        all_features = checkpoint_df.to_dict('records')
        print(f"📍 Resuming from checkpoint: frame {start_frame}/{total_frames}")
    else:
        start_frame  = 0
        all_features = []
        print(f"🆕 Starting fresh extraction")

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame - 1)
        ret, prev_frame = cap.read()
        if not ret:
            start_frame  = 0
            all_features = []
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, prev_frame = cap.read()
    else:
        ret, prev_frame = cap.read()

    if not ret:
        print("❌ Empty video")
        cap.release()
        return

    # FIX: white balance the very first frame before ROI crop
    prev_roi  = get_road_roi(grey_world_wb(prev_frame), session_ts)
    prev_gray = cv2.cvtColor(prev_roi, cv2.COLOR_BGR2GRAY)

    for frame_id in tqdm(range(start_frame if start_frame > 0 else 1, total_frames),
                         desc="Extracting",
                         initial=start_frame if start_frame > 0 else 1):
        ret, frame = cap.read()
        if not ret:
            break

        # FIX: Apply white balance ONCE per frame, then crop ROI
        # This ensures all downstream HSV features see colour-corrected pixels
        frame_wb  = grey_world_wb(frame)
        frame_roi = get_road_roi(frame_wb, session_ts)
        gray_roi  = cv2.cvtColor(frame_roi, cv2.COLOR_BGR2GRAY)

        features = {
            'frame_id': frame_id,
            'time_sec': frame_id / fps_actual
        }

        features.update(compute_optical_flow(prev_gray, gray_roi))
        features.update(compute_edge_features(gray_roi))
        features.update(compute_hsv_features(frame_roi))          # gets WB frame
        features.update(compute_wetness_index(frame_roi, gray_roi))  # gets WB frame
        features.update(compute_texture_features(gray_roi))
        features.update(compute_color_features(frame_roi))        # gets WB frame

        all_features.append(features)
        prev_gray = gray_roi

        if len(all_features) % CHECKPOINT_INTERVAL == 0:
            save_checkpoint(checkpoint_path, all_features)

    cap.release()

    if all_features:
        df = pd.DataFrame(all_features)
        df.to_csv(out_csv, index=False)
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        print(f"✅ Saved {len(df)} frames → {out_csv}")
        print(f"   Columns: {len(df.columns)} | mud_score present: {'mud_score' in df.columns}")

# -----------------------------
# MAIN LOOP
# -----------------------------
print("\n" + "="*60)
print("🚀 VISION FEATURE EXTRACTION — WITH WHITE BALANCE FIX")
print("="*60)
print("Fixes applied:")
print("  ✓ Grey-world white balance before HSV (fixes orange-light tiles → mud)")
print("  ✓ mud_score s<220 threshold (was s<150, missed real mud)")
print("  ✓ mud_score saved as explicit column")
print(f"\n💾 Checkpoints every {CHECKPOINT_INTERVAL} frames")
print("="*60)

for video in VIDEOS:
    process_video(video)

print("\n" + "="*60)
print("✅ ALL VIDEOS PROCESSED!")
print("Next: Cell 9 (IMU) → Cell 10 (Merger) → Cell 11 (R_skid) → Cell 12 (Report)")
print("="*60)

In [ ]:
# ============================================================
# CELL 9 — IMU Feature Extraction — Haversine GPS speed (SPEED_KMH never saved)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import os

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
BASE_DIR   = "/content/drive/MyDrive/data_collection"
SENSOR_DIR = f"{BASE_DIR}/sensor_data"
OUT_DIR    = f"{BASE_DIR}/imu_features"
WINDOW_SEC = 2.0
IMU_RATE   = 20.0

os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────
# HAVERSINE SPEED DERIVATION
# Computes speed in km/h between consecutive GPS fixes
# This is the ONLY reliable speed source — SPEED_KMH was
# never saved to any fusion CSV in any format
# ─────────────────────────────────────────
def haversine_kmh(lat1, lon1, lat2, lon2, dt_seconds):
    """
    Returns speed in km/h between two GPS coordinates
    separated by dt_seconds.
    """
    if dt_seconds <= 0:
        return 0.0
    R = 6371.0  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a    = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    dist_km = R * 2 * np.arcsin(np.sqrt(a))
    speed   = dist_km / (dt_seconds / 3600.0)  # km/h
    return float(speed)

def compute_gps_speeds(gps_df):
    """
    Given GPS rows with LAT, LON, TIME columns,
    compute speed_kmh for each row using distance to next fix.
    Returns a Series of speed values aligned to gps_df index.
    """
    gps = gps_df.copy().reset_index(drop=True)
    speeds = []

    for i in range(len(gps)):
        if i == 0:
            speeds.append(0.0)
            continue

        lat1 = gps.loc[i-1, "LAT"]
        lon1 = gps.loc[i-1, "LON"]
        lat2 = gps.loc[i,   "LAT"]
        lon2 = gps.loc[i,   "LON"]
        t1   = gps.loc[i-1, "TIME"]
        t2   = gps.loc[i,   "TIME"]

        # Skip if coordinates are missing or identical (no fix)
        if any(pd.isna([lat1, lon1, lat2, lon2])):
            speeds.append(0.0)
            continue
        if lat1 == lat2 and lon1 == lon2:
            # Same fix repeated — truly stationary or GPS frozen
            speeds.append(0.0)
            continue

        dt_sec = (t2 - t1).total_seconds()
        spd    = haversine_kmh(lat1, lon1, lat2, lon2, dt_sec)

        # Cap at 80 km/h — anything above is GPS noise/jump
        speeds.append(min(spd, 80.0))

    gps["SPEED_KMH_DERIVED"] = speeds
    return gps

# ─────────────────────────────────────────
# FEATURE EXTRACTION
# ─────────────────────────────────────────
def extract_accel_gyro_features(ax, ay, az, gx, gy, gz, dt):
    accel_mag = np.sqrt(ax**2 + ay**2 + az**2)
    gyro_mag  = np.sqrt(gx**2 + gy**2 + gz**2)
    return {
        "accel_mean"      : np.mean(accel_mag),
        "accel_std"       : np.std(accel_mag),
        "accel_max"       : np.max(accel_mag),
        "accel_z_var"     : np.var(az),
        "vibration_energy": np.sum(np.diff(accel_mag)**2),
        "jerk_mean"       : np.mean(np.abs(np.diff(accel_mag) / dt)),
        "jerk_max"        : np.max(np.abs(np.diff(accel_mag) / dt)),
        "gyro_mean"       : np.mean(gyro_mag),
        "gyro_std"        : np.std(gyro_mag),
        "gyro_max"        : np.max(gyro_mag),
        "roll_rate_mean"  : np.mean(np.abs(gx)),
        "roll_rate_std"   : np.std(gx),
        "pitch_rate_mean" : np.mean(np.abs(gy)),
        "pitch_rate_std"  : np.std(gy),
        "yaw_rate_mean"   : np.mean(np.abs(gz)),
        "stability_score" : np.std(accel_mag) + np.std(gyro_mag),
        "roughness_index" : np.sum(np.diff(accel_mag)**2) * np.var(gyro_mag),
    }

# ─────────────────────────────────────────
# EXTRACT FROM FUSION FILE
# ─────────────────────────────────────────
def extract_from_fusion(path):
    df = pd.read_csv(path)
    df = normalize_columns(df)
    df.columns = [c.upper() for c in df.columns]
    df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")
    df = df.dropna(subset=["TIME"]).sort_values("TIME")

    imu = df[df["TYPE"] == "IMU"].copy()
    if imu.empty:
        print(f"  ❌ No IMU rows"); return None

    required = ["AX", "AY", "AZ", "ROLL", "PITCH", "YAW"]
    missing  = [c for c in required if c not in imu.columns]
    if missing:
        print(f"  ❌ Missing columns: {missing}"); return None

    print(f"  ✓ {len(imu)} IMU samples")

    t0       = imu["TIME"].iloc[0]
    duration = (imu["TIME"].iloc[-1] - t0).total_seconds()
    n_wins   = int(duration // WINDOW_SEC)
    print(f"  ✓ Duration: {duration:.1f}s → {n_wins} windows")

    dt = 1.0 / IMU_RATE

    # ── Derive GPS speed from coordinates ────────────────────────────
    # SPEED_KMH was never saved in any fusion CSV format.
    # Compute from consecutive LAT/LON fixes using Haversine formula.
    gps_raw = df[df["TYPE"] == "GPS"].copy()
    if not gps_raw.empty and "LAT" in gps_raw.columns and "LON" in gps_raw.columns:
        gps_raw = gps_raw[
            gps_raw["LAT"].notna() & gps_raw["LON"].notna() &
            (gps_raw["LAT"] != 0)  & (gps_raw["LON"] != 0)
        ].copy()
        gps_rows = compute_gps_speeds(gps_raw) if not gps_raw.empty else pd.DataFrame()
    else:
        gps_rows = pd.DataFrame()

    has_speed = not gps_rows.empty and "SPEED_KMH_DERIVED" in gps_rows.columns
    if has_speed:
        nonzero_spd = (gps_rows["SPEED_KMH_DERIVED"] > 0).sum()
        print(f"  ✓ GPS speed derived: {nonzero_spd}/{len(gps_rows)} nonzero fixes")
    else:
        print(f"  ⚠ No GPS coordinates — speed will be 0")
    # ─────────────────────────────────────────────────────────────────

    all_features = []

    for wid in range(n_wins):
        t_start = wid * WINDOW_SEC
        t_end   = t_start + WINDOW_SEC

        mask = (
            (imu["TIME"] >= t0 + pd.Timedelta(seconds=t_start)) &
            (imu["TIME"] <  t0 + pd.Timedelta(seconds=t_end))
        )
        w = imu[mask]
        if len(w) < 10:
            continue

        ax = w["AX"].values
        ay = w["AY"].values
        az = w["AZ"].values

        roll_diff  = np.diff(w["ROLL"].values)
        pitch_diff = np.diff(w["PITCH"].values)
        yaw_diff   = np.diff(w["YAW"].values)

        gx = np.pad(roll_diff  / dt, (0, len(ax) - len(roll_diff)),  "edge")
        gy = np.pad(pitch_diff / dt, (0, len(ay) - len(pitch_diff)), "edge")
        gz = np.pad(yaw_diff   / dt, (0, len(az) - len(yaw_diff)),   "edge")

        feats = extract_accel_gyro_features(ax, ay, az, gx, gy, gz, dt)
        feats["window_id"]   = wid
        feats["t_start"]     = t_start
        feats["t_end"]       = t_end
        feats["imu_samples"] = len(w)

        # ── Assign derived GPS speed to this window ───────────────────
        # Try window match first, then nearest fix within 10 seconds
        if has_speed:
            gps_win = gps_rows[
                (gps_rows["TIME"] >= t0 + pd.Timedelta(seconds=t_start)) &
                (gps_rows["TIME"] <  t0 + pd.Timedelta(seconds=t_end))
            ]
            if gps_win.empty:
                t_mid     = t0 + pd.Timedelta(seconds=(t_start + t_end) / 2)
                diffs     = (gps_rows["TIME"] - t_mid).abs()
                near_idx  = diffs.idxmin()
                if diffs[near_idx].total_seconds() <= 10.0:
                    gps_win = gps_rows.loc[[near_idx]]

            if not gps_win.empty:
                feats["speed_kmh"] = float(
                    gps_win["SPEED_KMH_DERIVED"].median()
                )
            else:
                feats["speed_kmh"] = 0.0
        else:
            feats["speed_kmh"] = 0.0
        # ─────────────────────────────────────────────────────────────

        all_features.append(feats)

    return pd.DataFrame(all_features) if all_features else None

# ─────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────
fusion_files = sorted(Path(SENSOR_DIR).glob("fusion_*.csv"))
print(f"Found {len(fusion_files)} fusion files\n")
print("Method: GPS speed DERIVED from Haversine formula")
print("  (SPEED_KMH never saved to fusion CSV in any format)\n")

for path in fusion_files:
    ts = path.stem.replace("fusion_", "")
    print(f"\n{'='*60}")
    print(f"▶ {path.name}")
    print(f"{'='*60}")

    features_df = extract_from_fusion(path)

    if features_df is not None and len(features_df) > 0:
        out = f"{OUT_DIR}/imu_features_{ts}.csv"
        features_df.to_csv(out, index=False)
        print(f"  ✅ Saved {len(features_df)} windows → {out}")
        zero_spd   = (features_df["speed_kmh"] == 0).sum()
        nonzero    = (features_df["speed_kmh"] > 0).sum()
        max_spd    = features_df["speed_kmh"].max()
        print(f"  Speed: {nonzero} moving windows | "
              f"{zero_spd} stationary | max={max_spd:.1f} km/h")
    else:
        print(f"  ❌ No features extracted")

print(f"\n✅ IMU extraction complete → {OUT_DIR}")

In [ ]:
# ============================================================
# CELL 10 — Merger: vision + IMU → label_helpers  (recompute_wetness fix + brightness_variance/mud_score passthrough)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import os

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
BASE_DIR   = "/content/drive/MyDrive/data_collection"
VISION_DIR = f"{BASE_DIR}/video_features"
IMU_DIR    = f"{BASE_DIR}/imu_features"
OUT_DIR    = f"{BASE_DIR}/label_helpers"
WINDOW_SEC = 2.0

os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────
# WETNESS RECOMPUTE  —  CRITICAL FIX
# The per-frame wetness_ratio saved in Cell 8 uses loose thresholds
# (wet_tar s<50 v>120, water s<30 v>150) that flag dry grey asphalt
# as wet. Recompute here with tightened thresholds and apply BEFORE
# aggregating, so both the merged label_helpers and the downstream
# R_skid vision term (Cell 11) use corrected wetness.
# ─────────────────────────────────────────
def recompute_wetness(df):
    s = df['saturation_mean']
    v = df['value_mean']
    h = df['hue_mean']
    wet_tar   = ((s < 30) & (v > 180)).astype(float)
    wet_water = ((s < 15) & (v > 200)).astype(float)
    wet_mud   = ((h > 10) & (h < 40) & (s > 40) & (s < 220) & (v > 80) & (v < 210)).astype(float)
    return pd.concat([wet_tar, wet_water, wet_mud], axis=1).max(axis=1)

# ─────────────────────────────────────────
# VISION AGGREGATION  (per-frame → per-window)
# ─────────────────────────────────────────
def aggregate_vision_to_windows(vision_df, window_sec):
    # CRITICAL FIX: recompute wetness with tightened thresholds BEFORE
    # aggregating (overrides the loose per-frame wetness_ratio from Cell 8)
    vision_df = vision_df.copy()
    vision_df["wetness_ratio"] = recompute_wetness(vision_df)

    vision_df["window_id"] = (vision_df["time_sec"] // window_sec).astype(int)
    aggregated = []
    for wid, group in vision_df.groupby("window_id"):
        w = {
            "window_id"            : wid,
            "t_start"              : wid * window_sec,
            "t_end"                : (wid + 1) * window_sec,
            "frame_count"          : len(group),
            "flow_mean"            : group["flow_mag_mean"].mean(),
            "flow_std"             : group["flow_mag_mean"].std(),
            "flow_max"             : group["flow_mag_mean"].max(),
            "flow_variance"        : group["flow_mag_mean"].var(),
            "edge_density_mean"    : group["edge_density"].mean(),
            "edge_magnitude_mean"  : group["edge_magnitude_mean"].mean(),
            "hue_mean"             : group["hue_mean"].mean(),
            "saturation_mean"      : group["saturation_mean"].mean(),
            "value_mean"           : group["value_mean"].mean(),
            "wetness_ratio"        : group["wetness_ratio"].mean(),
            "specular_ratio"       : group["specular_ratio"].mean(),
            "brightness_uniformity": group["brightness_uniformity"].mean(),
            "texture_roughness"    : group["texture_roughness"].mean(),
            "lightness_mean"       : group["lightness_mean"].mean(),
            "color_variance"       : group["color_variance"].mean(),
        }
        # Pass through extra vision features when Cell 8 saved them
        if "brightness_variance" in group.columns:
            w["brightness_variance"] = group["brightness_variance"].mean()
        if "mud_score" in group.columns:
            w["mud_score"] = group["mud_score"].mean()

        w["roughness_index"] = w["flow_variance"] * w["edge_density_mean"]
        w["wet_indicator"]   = w["wetness_ratio"] * w["specular_ratio"]
        aggregated.append(w)
    return pd.DataFrame(aggregated)

# ─────────────────────────────────────────
# MAIN LOOP — match vision files to IMU files by timestamp
# ─────────────────────────────────────────
vision_files = sorted(Path(VISION_DIR).glob("video_*_features.csv"))
print(f"Found {len(vision_files)} vision feature files\n")

ok = 0; vision_only = 0; failed = 0

for vpath in vision_files:
    # Extract timestamp from filename: video_20260103130421_features.csv
    ts = vpath.stem.replace("video_", "").replace("_features", "")

    print(f"\n{'='*60}")
    print(f"▶ {ts}")
    print(f"{'='*60}")

    try:
        vision_raw      = pd.read_csv(vpath)
        vision_windowed = aggregate_vision_to_windows(vision_raw, WINDOW_SEC)
        print(f"  ✓ Vision: {len(vision_raw)} frames → {len(vision_windowed)} windows")
    except Exception as e:
        print(f"  ❌ Vision load failed: {e}"); failed += 1; continue

    imu_path = Path(f"{IMU_DIR}/imu_features_{ts}.csv")

    if imu_path.exists():
        try:
            imu_df = pd.read_csv(imu_path)
            print(f"  ✓ IMU: {len(imu_df)} windows")

            merged = pd.merge(
                vision_windowed, imu_df,
                on="window_id", how="left",
                suffixes=("", "_imu")
            )
            # Drop duplicate time columns from IMU side
            for col in ["t_start_imu", "t_end_imu"]:
                if col in merged.columns:
                    merged.drop(columns=[col], inplace=True)

            imu_coverage = merged["imu_samples"].notna().sum()
            print(f"  ✓ Merged: {len(merged)} windows ({imu_coverage} with IMU)")
            data_type = "multimodal"
        except Exception as e:
            print(f"  ⚠️  IMU merge failed: {e} → vision-only")
            merged = vision_windowed; data_type = "vision_only"; vision_only += 1
    else:
        print(f"  ⚠️  No IMU features found → vision-only")
        merged = vision_windowed; data_type = "vision_only"; vision_only += 1

    merged["data_type"] = data_type

    out_path = f"{OUT_DIR}/label_helper_{ts}.csv"
    merged.to_csv(out_path, index=False)
    print(f"  ✅ Saved: {out_path} ({len(merged)} windows, {len(merged.columns)} cols)")
    ok += 1

print(f"\n{'='*60}")
print(f"✅ Done: {ok} processed, {vision_only} vision-only, {failed} failed")
print(f"Output: {OUT_DIR}")

In [ ]:
# ============================================================
# CELL 11 — R_skid Computation + GPS Heatmap — vision GLOBAL / IMU+gyro PER-SESSION norm; motion gate & speed mult DISABLED
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from pathlib import Path
from IPython.display import display
import os

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/data_collection"
HELPERS_DIR = f"{BASE_DIR}/label_helpers"
SENSOR_DIR  = f"{BASE_DIR}/sensor_data"
OUT_DIR     = f"{BASE_DIR}/rskid_output"
os.makedirs(OUT_DIR, exist_ok=True)

WINDOW_SEC        = 2.0
LAMBDA            = 0.3
MAX_SPEED         = 60.0
SAFE_MAX          = 0.33
CAUTION_MAX       = 0.66
STATIONARY_THRESH = 2.0

# ─────────────────────────────────────────
# STEP 1 — LOAD ALL LABEL HELPERS
# ─────────────────────────────────────────
all_files = sorted(Path(HELPERS_DIR).glob("label_helper_*.csv"))
all_dfs = []
for f in all_files:
    df = pd.read_csv(f)
    df.columns = [c.upper() for c in df.columns]
    df["SESSION"] = f.stem.replace("label_helper_", "")
    all_dfs.append(df)

if not all_dfs:
    raise RuntimeError("No label helpers found.")

full_df = pd.concat(all_dfs, ignore_index=True)
full_df.columns = [c.upper() for c in full_df.columns]
print(f"Loaded {len(full_df)} windows across {full_df['SESSION'].nunique()} sessions\n")

# ─────────────────────────────────────────
# STEP 2 — RAW TERMS
# ─────────────────────────────────────────

# Term 1: IMU — Z-axis acceleration variance
if "ACCEL_Z_VAR" in full_df.columns:
    term_imu_raw = full_df["ACCEL_Z_VAR"].fillna(0)
    print("✓ Term 1: ACCEL_Z_VAR")
else:
    term_imu_raw = pd.Series(np.zeros(len(full_df)), index=full_df.index)
    print("⚠ Term 1: zero (no IMU data)")

# Term 2: Gyro — roll + yaw rate
if "ROLL_RATE_MEAN" in full_df.columns and "YAW_RATE_MEAN" in full_df.columns:
    term_gyro_raw = (
        full_df["ROLL_RATE_MEAN"].fillna(0) +
        full_df["YAW_RATE_MEAN"].fillna(0)
    )
    print("✓ Term 2: ROLL_RATE_MEAN + YAW_RATE_MEAN")
else:
    term_gyro_raw = pd.Series(np.zeros(len(full_df)), index=full_df.index)
    print("⚠ Term 2: zero (no gyro data)")

# Term 3: Vision — simple mean of available columns
# NOTE: TERM_VISION uses ONLY these 3 bounded features.
# BRIGHTNESS_VARIANCE (raw pixel variance, range ~0-1644) and MUD_SCORE are
# still passed through to label_helpers / rskid_all_sessions.csv for analysis,
# but are EXCLUDED from the score. They are averaged un-weighted and then
# global min-max normalized; brightness_variance's huge scale dominates the
# mean and collapses every real window toward 0 (this is what flattened
# R_skid to mean ~0.13 / almost-all-Safe). Keep the vision term on the
# validated 0.42-8.14 scale.
vision_candidates     = ["WETNESS_RATIO", "EDGE_DENSITY_MEAN", "TEXTURE_ROUGHNESS"]
vision_cols_available = [c for c in vision_candidates if c in full_df.columns]
print(f"✓ Term 3: vision from {vision_cols_available}")
term_vision_raw = full_df[vision_cols_available].fillna(0).mean(axis=1)

print(f"\nRaw ranges:")
print(f"  vision : {term_vision_raw.min():.4f} – {term_vision_raw.max():.4f}")
print(f"  imu    : {term_imu_raw.min():.4f} – {term_imu_raw.max():.4f}  nonzero={(term_imu_raw>0).sum()}/{len(term_imu_raw)}")
print(f"  gyro   : {term_gyro_raw.min():.4f} – {term_gyro_raw.max():.4f}  nonzero={(term_gyro_raw>0).sum()}/{len(term_gyro_raw)}")

# ─────────────────────────────────────────
# STEP 3 — NORMALIZATION
# Vision  → GLOBAL min-max across all sessions
#   Preserves real cross-session differences:
#   clean dry road (0.42) vs muddy road (8.14) must NOT collapse to same range
# IMU/Gyro → PER-SESSION min-max
#   Removes hardware/mounting bias (ACCEL_Z_VAR range 0.00006–17585)
#   while preserving within-session roughness contrast
# ─────────────────────────────────────────
def minmax_norm_global(raw_series):
    mn, mx = raw_series.min(), raw_series.max()
    if (mx - mn) < 1e-9:
        return pd.Series(np.zeros(len(raw_series)), index=raw_series.index)
    return ((raw_series - mn) / (mx - mn)).clip(0, 1)

def minmax_norm_per_session(full_df, raw_series):
    result = raw_series.copy().astype(float)
    for session_ts, group in full_df.groupby("SESSION"):
        idx  = group.index
        vals = raw_series.loc[idx]
        mn, mx = vals.min(), vals.max()
        if (mx - mn) < 1e-9:
            result.loc[idx] = 0.0
        else:
            result.loc[idx] = (vals - mn) / (mx - mn)
    return result

term_vision = minmax_norm_global(term_vision_raw)
term_imu    = minmax_norm_per_session(full_df, term_imu_raw)
term_gyro   = minmax_norm_per_session(full_df, term_gyro_raw)

print(f"\nNormalized ranges:")
print(f"  term_vision (global) : {term_vision.min():.4f} – {term_vision.max():.4f}")
print(f"  term_imu   (per-sess): {term_imu.min():.4f} – {term_imu.max():.4f}")
print(f"  term_gyro  (per-sess): {term_gyro.min():.4f} – {term_gyro.max():.4f}")

full_df["TERM_IMU"]    = term_imu.values
full_df["TERM_GYRO"]   = term_gyro.values
full_df["TERM_VISION"] = term_vision.values

# ─────────────────────────────────────────
# STEP 4 — R_SKID BASE
# Sessions WITH IMU : standard weights 0.5 / 0.3 / 0.2
# Sessions WITHOUT IMU : vision carries full weight
#   (no-IMU sessions are all 2025 sessions — without this fix
#    they max out at 0.5 × vision = 0.5, never reaching Caution)
# ─────────────────────────────────────────
has_imu = (full_df["TERM_IMU"] > 0) | (full_df["TERM_GYRO"] > 0)

full_df["R_SKID_BASE"] = (
    0.5 * full_df["TERM_VISION"] +
    0.3 * full_df["TERM_IMU"]    +
    0.2 * full_df["TERM_GYRO"]
)

# No-IMU windows: vision = sole signal, rescale to full [0,1]
full_df.loc[~has_imu, "R_SKID_BASE"] = full_df.loc[~has_imu, "TERM_VISION"]

print(f"\n✓ R_SKID_BASE computed")
print(f"  Windows with IMU   : {has_imu.sum()} ({has_imu.sum()/len(full_df)*100:.1f}%)")
print(f"  Windows without IMU: {(~has_imu).sum()} ({(~has_imu).sum()/len(full_df)*100:.1f}%)")

# ─────────────────────────────────────────
# STEP 5 — MOTION GATE (DISABLED)
# Disabled because SPEED_KMH was never saved to fusion CSVs.
# GPS speed is now derived via Haversine in Cell 9.
# Re-enable after confirming Cell 11 produces real speeds
# (look for "Speed: X moving windows" > 0 in Cell 9 output).
# ─────────────────────────────────────────
print("⚠ Motion gate disabled — enable after Cell 9 GPS speed confirmed")

# ─────────────────────────────────────────
# STEP 6 — SPEED MULTIPLIER (DISABLED)
# Disabled for same reason as motion gate above.
# R_SKID_FINAL = R_SKID_BASE until speed is confirmed working.
# ─────────────────────────────────────────
full_df["SPEED_NORM"]   = 0.0
full_df["R_SKID_FINAL"] = full_df["R_SKID_BASE"].clip(0, 1)
print("⚠ Speed multiplier disabled — R_SKID_FINAL = R_SKID_BASE")

# ─────────────────────────────────────────
# STEP 7 — RISK LABELS
# ─────────────────────────────────────────
def assign_risk_class(score):
    if score < SAFE_MAX:      return "Safe"
    elif score < CAUTION_MAX: return "Caution"
    else:                     return "High Risk"

full_df["RISK_CLASS"] = full_df["R_SKID_FINAL"].apply(assign_risk_class)

print("\n── Risk Class Distribution ──")
print(full_df["RISK_CLASS"].value_counts())
print(f"\nMean R_skid : {full_df['R_SKID_FINAL'].mean():.3f}")
print(f"Max  R_skid : {full_df['R_SKID_FINAL'].max():.3f}")

print("\n── Per-Session Breakdown ──")
for session_ts, group in full_df.groupby("SESSION"):
    counts  = group["RISK_CLASS"].value_counts()
    safe    = counts.get("Safe", 0)
    caution = counts.get("Caution", 0)
    high    = counts.get("High Risk", 0)
    mean_r  = group["R_SKID_FINAL"].mean()
    print(f"  {session_ts} | windows: {len(group):4d} | "
          f"mean R_skid: {mean_r:.3f} | "
          f"Safe: {safe/len(group)*100:.0f}% "
          f"Caution: {caution/len(group)*100:.0f}% "
          f"High: {high/len(group)*100:.0f}%")

# ─────────────────────────────────────────
# STEP 8 — SCORE DISTRIBUTION PLOTS
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("R_skid Score Distributions", fontsize=14)

axes[0].hist(full_df["R_SKID_BASE"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(SAFE_MAX,    color="green", linestyle="--", label="Safe/Caution")
axes[0].axvline(CAUTION_MAX, color="red",   linestyle="--", label="Caution/High")
axes[0].set_title("R_skid Base"); axes[0].set_xlabel("Score")
axes[0].legend(); axes[0].grid()

axes[1].hist(full_df["R_SKID_FINAL"], bins=40, color="darkorange", edgecolor="white")
axes[1].axvline(SAFE_MAX,    color="green", linestyle="--", label="Safe/Caution")
axes[1].axvline(CAUTION_MAX, color="red",   linestyle="--", label="Caution/High")
axes[1].set_title("R_skid Final"); axes[1].set_xlabel("Score")
axes[1].legend(); axes[1].grid()

class_counts = full_df["RISK_CLASS"].value_counts().reindex(
    ["Safe", "Caution", "High Risk"], fill_value=0)
axes[2].bar(class_counts.index, class_counts.values,
            color=["green", "orange", "red"])
axes[2].set_title("Windows per Risk Class")
axes[2].set_ylabel("Count"); axes[2].grid(axis="y")
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────
# STEP 9 — TERM CONTRIBUTION PLOT
# ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(full_df.index, full_df["TERM_IMU"],    label="Term 1: IMU",    alpha=0.7)
ax.plot(full_df.index, full_df["TERM_GYRO"],   label="Term 2: Gyro",   alpha=0.7)
ax.plot(full_df.index, full_df["TERM_VISION"], label="Term 3: Vision", alpha=0.7)
ax.plot(full_df.index, full_df["R_SKID_FINAL"], label="R_skid Final",
        color="black", linewidth=1.5)
ax.axhline(SAFE_MAX,    color="green", linestyle="--", alpha=0.5)
ax.axhline(CAUTION_MAX, color="red",   linestyle="--", alpha=0.5)
ax.set_title("R_skid Terms and Final Score")
ax.set_xlabel("Window Index"); ax.set_ylabel("Normalized Score [0–1]")
ax.legend(); ax.grid()
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────
# STEP 10 — SANITY CHECK TABLE
# ─────────────────────────────────────────
sanity_cols = [c for c in ["SESSION", "WINDOW_ID", "T_START",
                            "R_SKID_FINAL", "RISK_CLASS",
                            "TERM_IMU", "TERM_GYRO", "TERM_VISION",
                            "SPEED_KMH", "WETNESS_RATIO"]
               if c in full_df.columns]

print("\n── Top 10 Highest Risk Windows ──")
display(full_df.nlargest(10, "R_SKID_FINAL")[sanity_cols].round(4))

print("\n── Top 10 Lowest Risk Windows ──")
display(full_df.nsmallest(10, "R_SKID_FINAL")[sanity_cols].round(4))

# ─────────────────────────────────────────
# STEP 11 — GPS HEATMAP
# ─────────────────────────────────────────
def get_gps_for_windows(session_ts, window_ids, window_sec):
    fusion_path = Path(f"{SENSOR_DIR}/fusion_{session_ts}.csv")
    if not fusion_path.exists():
        return {}
    df = pd.read_csv(fusion_path)
    df.columns = [c.upper() for c in df.columns]
    if "D1" in df.columns and "LAT" not in df.columns:
        gps_mask = df["TYPE"] == "GPS"
        df.loc[gps_mask, "LAT"] = pd.to_numeric(df.loc[gps_mask, "D1"], errors="coerce")
        df.loc[gps_mask, "LON"] = pd.to_numeric(df.loc[gps_mask, "D2"], errors="coerce")
    df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")
    df = df.dropna(subset=["TIME"]).sort_values("TIME")
    gps = df[df["TYPE"] == "GPS"].copy()
    gps = gps[gps["LAT"].notna() & gps["LON"].notna() &
              (gps["LAT"] != 0)  & (gps["LON"] != 0)]
    if gps.empty:
        return {}
    t0 = df["TIME"].iloc[0]
    result = {}
    for wid in window_ids:
        t_start = wid * window_sec
        t_end   = t_start + window_sec
        mask = ((gps["TIME"] >= t0 + pd.Timedelta(seconds=t_start)) &
                (gps["TIME"] <  t0 + pd.Timedelta(seconds=t_end)))
        gps_win = gps[mask]
        if not gps_win.empty:
            result[wid] = (gps_win["LAT"].mean(), gps_win["LON"].mean())
    return result

heatmap_rows = []
for session_ts, group in full_df.groupby("SESSION"):
    gps_map = get_gps_for_windows(
        session_ts, group["WINDOW_ID"].tolist(), WINDOW_SEC
    )
    for _, row in group.iterrows():
        wid = row["WINDOW_ID"]
        if wid in gps_map:
            lat, lon = gps_map[wid]
            heatmap_rows.append({
                "lat"       : lat,
                "lon"       : lon,
                "r_skid"    : row["R_SKID_FINAL"],
                "risk_class": row["RISK_CLASS"],
                "session"   : session_ts,
                "speed_kmh" : row.get("SPEED_KMH", 0),
            })

if heatmap_rows:
    heatmap_df = pd.DataFrame(heatmap_rows)
    print(f"\n✅ GPS heatmap: {len(heatmap_df)} points across "
          f"{heatmap_df['session'].nunique()} sessions")

    lat_c = heatmap_df["lat"].mean()
    lon_c = heatmap_df["lon"].mean()
    m = folium.Map(location=[lat_c, lon_c], zoom_start=16, tiles="OpenStreetMap")
    color_map = {"Safe": "green", "Caution": "orange", "High Risk": "red"}

    for _, row in heatmap_df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            color=color_map.get(row["risk_class"], "blue"),
            fill=True,
            fill_opacity=0.75,
            popup=folium.Popup(
                f"R_skid: {row['r_skid']:.3f}<br>"
                f"Class: {row['risk_class']}<br>"
                f"Speed: {row['speed_kmh']:.1f} km/h<br>"
                f"Session: {row['session']}",
                max_width=200
            )
        ).add_to(m)

    display(m)
    m.save(f"{OUT_DIR}/risk_heatmap.html")
    print(f"✅ Heatmap saved → {OUT_DIR}/risk_heatmap.html")
else:
    print("⚠ No GPS data found — heatmap skipped")

# ─────────────────────────────────────────
# STEP 12 — SAVE FULL DATASET
# ─────────────────────────────────────────
out_csv = f"{OUT_DIR}/rskid_all_sessions.csv"
full_df.to_csv(out_csv, index=False)
print(f"\n✅ Saved {len(full_df)} windows → {out_csv}")
print(f"   Columns: {len(full_df.columns)}")
print(f"\nNext: Run Cell 12 for analysis report")

In [ ]:
# ============================================================
# CELL 12 — Analysis Report + Figures
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf
from pathlib import Path
import os

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/data_collection"
OUT_DIR  = f"{BASE_DIR}/rskid_output"
os.makedirs(OUT_DIR, exist_ok=True)

# Load the saved R_skid dataset from Cell 11
full_df = pd.read_csv(f"{OUT_DIR}/rskid_all_sessions.csv")
full_df.columns = [c.upper() for c in full_df.columns]

print(f"Loaded {len(full_df)} windows across {full_df['SESSION'].nunique()} sessions")

# ─────────────────────────────────────────
# FIGURE 1 — Score distributions (save)
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("R_skid Score Distributions — All Sessions", fontsize=14)

axes[0].hist(full_df["R_SKID_BASE"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(0.33, color="green", linestyle="--", label="Safe/Caution")
axes[0].axvline(0.66, color="red",   linestyle="--", label="Caution/High")
axes[0].set_title("R_skid Base"); axes[0].set_xlabel("Score")
axes[0].legend(); axes[0].grid()

axes[1].hist(full_df["R_SKID_FINAL"], bins=40, color="darkorange", edgecolor="white")
axes[1].axvline(0.33, color="green", linestyle="--", label="Safe/Caution")
axes[1].axvline(0.66, color="red",   linestyle="--", label="Caution/High")
axes[1].set_title("R_skid Final (×Speed)"); axes[1].set_xlabel("Score")
axes[1].legend(); axes[1].grid()

class_counts = full_df["RISK_CLASS"].value_counts().reindex(
    ["Safe", "Caution", "High Risk"], fill_value=0
)
axes[2].bar(class_counts.index, class_counts.values,
            color=["green", "orange", "red"])
axes[2].set_title("Windows per Risk Class")
axes[2].set_ylabel("Count"); axes[2].grid(axis="y")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig1_score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: fig1_score_distributions.png")

# ─────────────────────────────────────────
# FIGURE 2 — Term contributions (save)
# ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(full_df.index, full_df["TERM_IMU"],    label="Term 1: IMU (σ²az)",      alpha=0.7)
ax.plot(full_df.index, full_df["TERM_GYRO"],   label="Term 2: Gyro (roll+yaw)", alpha=0.7)
ax.plot(full_df.index, full_df["TERM_VISION"], label="Term 3: Vision",           alpha=0.7)
ax.plot(full_df.index, full_df["R_SKID_FINAL"],label="R_skid Final",
        color="black", linewidth=1.5)
ax.axhline(0.33, color="green", linestyle="--", alpha=0.5)
ax.axhline(0.66, color="red",   linestyle="--", alpha=0.5)
ax.set_title("R_skid Term Contributions Over All Windows")
ax.set_xlabel("Window Index"); ax.set_ylabel("Normalized Score [0–1]")
ax.legend(); ax.grid()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_term_contributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: fig2_term_contributions.png")

# ─────────────────────────────────────────
# FIGURE 3 — Per-session breakdown (save)
# ─────────────────────────────────────────
session_stats = full_df.groupby("SESSION").agg(
    total_windows   = ("R_SKID_FINAL", "count"),
    mean_rskid      = ("R_SKID_FINAL", "mean"),
    max_rskid       = ("R_SKID_FINAL", "max"),
    safe_pct        = ("RISK_CLASS",   lambda x: (x=="Safe").mean() * 100),
    caution_pct     = ("RISK_CLASS",   lambda x: (x=="Caution").mean() * 100),
    highrisk_pct    = ("RISK_CLASS",   lambda x: (x=="High Risk").mean() * 100),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Per-Session R_skid Summary", fontsize=13)

# Mean R_skid per session
axes[0].barh(session_stats["SESSION"], session_stats["mean_rskid"], color="steelblue")
axes[0].axvline(0.33, color="green", linestyle="--")
axes[0].axvline(0.66, color="red",   linestyle="--")
axes[0].set_title("Mean R_skid per Session")
axes[0].set_xlabel("Mean R_skid Final"); axes[0].grid(axis="x")

# Stacked risk class percentage per session
axes[1].barh(session_stats["SESSION"], session_stats["safe_pct"],
             color="green", label="Safe")
axes[1].barh(session_stats["SESSION"], session_stats["caution_pct"],
             left=session_stats["safe_pct"],
             color="orange", label="Caution")
axes[1].barh(session_stats["SESSION"], session_stats["highrisk_pct"],
             left=session_stats["safe_pct"] + session_stats["caution_pct"],
             color="red", label="High Risk")
axes[1].set_title("Risk Class Distribution per Session")
axes[1].set_xlabel("Percentage of Windows")
axes[1].legend(); axes[1].grid(axis="x")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig3_per_session_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: fig3_per_session_breakdown.png")

# ─────────────────────────────────────────
# SUMMARY REPORT — plain text
# ─────────────────────────────────────────
report_lines = [
    "=" * 60,
    "SafeRHD — R_skid Analysis Report",
    "=" * 60,
    f"Total sessions     : {full_df['SESSION'].nunique()}",
    f"Total windows      : {len(full_df)}",
    f"Total data duration: {len(full_df) * 2 / 60:.1f} minutes",
    "",
    "── Risk Class Distribution ──",
    f"  Safe      : {(full_df['RISK_CLASS']=='Safe').sum():4d} windows "
    f"({(full_df['RISK_CLASS']=='Safe').mean()*100:.1f}%)",
    f"  Caution   : {(full_df['RISK_CLASS']=='Caution').sum():4d} windows "
    f"({(full_df['RISK_CLASS']=='Caution').mean()*100:.1f}%)",
    f"  High Risk : {(full_df['RISK_CLASS']=='High Risk').sum():4d} windows "
    f"({(full_df['RISK_CLASS']=='High Risk').mean()*100:.1f}%)",
    "",
    "── R_skid Score Statistics ──",
    f"  Mean R_skid base  : {full_df['R_SKID_BASE'].mean():.4f}",
    f"  Mean R_skid final : {full_df['R_SKID_FINAL'].mean():.4f}",
    f"  Max  R_skid final : {full_df['R_SKID_FINAL'].max():.4f}",
    f"  Std  R_skid final : {full_df['R_SKID_FINAL'].std():.4f}",
    "",
    "── Term Contributions (mean across all windows) ──",
    f"  Term 1 IMU    : {full_df['TERM_IMU'].mean():.4f}",
    f"  Term 2 Gyro   : {full_df['TERM_GYRO'].mean():.4f}",
    f"  Term 3 Vision : {full_df['TERM_VISION'].mean():.4f}",
    "",
    "── Per Session Summary ──",
]

for _, row in session_stats.iterrows():
    report_lines.append(
        f"  {row['SESSION']} | windows: {int(row['total_windows']):4d} | "
        f"mean R_skid: {row['mean_rskid']:.3f} | "
        f"Safe: {row['safe_pct']:.0f}% "
        f"Caution: {row['caution_pct']:.0f}% "
        f"High: {row['highrisk_pct']:.0f}%"
    )

report_lines += [
    "",
    "── Output Files ──",
    f"  rskid_all_sessions.csv      — full dataset with R_skid scores",
    f"  risk_heatmap.html           — GPS risk heatmap (open in browser)",
    f"  fig1_score_distributions.png",
    f"  fig2_term_contributions.png",
    f"  fig3_per_session_breakdown.png",
    "=" * 60,
]

report_text = "\n".join(report_lines)
print(report_text)

report_path = f"{OUT_DIR}/rskid_analysis_report.txt"
with open(report_path, "w") as f:
    f.write(report_text)
print(f"\n✅ Report saved: {report_path}")
print(f"\nAll outputs in: {OUT_DIR}")